# GCQ Attack – Model Setup

Environment, model loading, generation configuration, and token filtering utilities.

## Model loading

Loads the Vicuna model and tokenizer.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
tokenizer = AutoTokenizer.from_pretrained("lmsys/vicuna-7b-v1.3")
model = AutoModelForCausalLM.from_pretrained(
    "lmsys/vicuna-7b-v1.3",
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()

## Generation config

Shared generation settings for completion generation.

In [ ]:
from transformers import GenerationConfig


gen_cfg = GenerationConfig(
    bos_token_id=1,
    eos_token_id=2,
    pad_token_id=0,
    max_new_tokens=500,
    temperature=0.7,
    top_k=None,
    top_p=0.9,
    do_sample=True
)


## Vocabulary filtering

Builds a list of allowed English token ids using NLTK words.

In [ ]:
import re
import nltk
from nltk.corpus import words

nltk.download('words')
english_vocab = set(w.lower() for w in words.words())

def get_english_token_ids(tokenizer):
    allowed = []
    for tok_id in range(tokenizer.vocab_size):
        decoded = tokenizer.decode([tok_id]).strip()

        # Skip empty or non-printable
        if not decoded or not all(32 <= ord(c) < 127 for c in decoded):
            continue

        # Extract the actual word part (some tokens may include punctuation etc.)
        word_match = re.match(r"^[a-zA-Z]+$", decoded)
        if word_match and decoded.lower() in english_vocab:
            allowed.append(tok_id)
    return allowed

allowed_token_ids = get_english_token_ids(tokenizer)

In [ ]:
print(len(allowed_token_ids))